# 05 — Geographic Sales Analysis

**Goal:** Visualise revenue, order volume, and customer distribution
across Brazilian states. Build a choropleth map.

---

In [ ]:
import sys
sys.path.append('../src')

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
import seaborn as sns

# Optional — install with: pip install folium
try:
    import folium
    FOLIUM = True
except ImportError:
    FOLIUM = False
    print('folium not installed — skipping interactive map.')

from analysis import set_style, revenue_by_state, plot_revenue_by_state, PALETTE

set_style()
print('Ready ✅')

In [ ]:
master = pd.read_csv('../data/processed/master.csv', parse_dates=['order_purchase_timestamp'])
print(f'Loaded {master.shape[0]:,} rows')

## 1. Revenue by State Table

In [ ]:
state_df = revenue_by_state(master)
state_df.style.format({'revenue': 'R${:,.2f}', 'avg_order_value': 'R${:,.2f}'})

## 2. Bar Chart — Revenue by State (Top 15)

In [ ]:
plot_revenue_by_state(master, top_n=15, save_path='../images/revenue_by_state.png')

## 3. Customers vs Revenue Scatter by State

In [ ]:
fig, ax = plt.subplots(figsize=(10, 7))

ax.scatter(state_df['customers'], state_df['revenue'],
           s=state_df['avg_order_value'] * 2,
           color=PALETTE['primary'], alpha=0.65, edgecolors='white', linewidths=0.8)

for _, row in state_df.iterrows():
    ax.annotate(row['customer_state'],
                (row['customers'], row['revenue']),
                fontsize=8, ha='center', va='bottom')

ax.set_xlabel('Number of Customers')
ax.set_ylabel('Revenue (R$)')
ax.yaxis.set_major_formatter(mticker.FuncFormatter(lambda x, _: f'R${x/1e6:.1f}M'))
ax.set_title('Customers vs Revenue by State\n(bubble size = avg order value)',
             fontsize=13, fontweight='bold')
plt.tight_layout()
plt.savefig('../images/state_scatter.png', dpi=150, bbox_inches='tight')
plt.show()

## 4. Interactive Folium Choropleth Map
_Requires `folium` and the Brazil GeoJSON. Run `pip install folium` if not installed._

In [ ]:
if FOLIUM:
    import requests, json

    # Download Brazil state GeoJSON
    geojson_url = 'https://raw.githubusercontent.com/codeforamerica/click_that_hood/master/public/data/brazil-states.geojson'
    brazil_geo = requests.get(geojson_url).json()

    # Fix state abbreviation key — GeoJSON uses 'sigla'
    m = folium.Map(location=[-14.2, -51.9], zoom_start=4, tiles='CartoDB positron')

    folium.Choropleth(
        geo_data=brazil_geo,
        name='Revenue by State',
        data=state_df,
        columns=['customer_state', 'revenue'],
        key_on='feature.properties.sigla',
        fill_color='YlOrRd',
        fill_opacity=0.75,
        line_opacity=0.3,
        legend_name='Revenue (R$)',
    ).add_to(m)

    folium.LayerControl().add_to(m)
    m.save('../images/brazil_choropleth.html')
    print('✅ Map saved → images/brazil_choropleth.html')
    m
else:
    print('Install folium to see the interactive map.')